# NUTDTS 816 Time Series Analysis
## L21 Returns, stylised facts, GARCH

Lab notebook for Chapter 11 of the lecture notes. Run the setup cell first. Every code cell reproduces an example from the notes; the exercises at the end are from the chapter's self-check list.

**Instructor:** Dr Tolulope Adesina · NUTM MSc Data Science · 2026

In [ ]:
# ---- Setup: run once per Colab session ----
# 1. Install the course libraries (about two minutes the first time)
!pip install -q statsmodels pmdarima statsforecast neuralforecast lightgbm arch plotly

# 2. Fetch the course data module and data snapshots from the course repository.
#    Replace REPO with your fork or the official course repository URL.
REPO = "https://raw.githubusercontent.com/<your-github-user>/nutdts816/main"
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(f"{REPO}/src/tsdata.py", "tsdata.py")
DATA_FILES = ["nigeria_cpi", "nigeria_fx", "nigeria_grid", "nigeria_malaria", "nigeria_rainfall", "bonny_light", "daily_demand",
              "airpassengers", "a10", "h02", "ausbeer", "elecequip", "usmelec", "goog", "nile", "austourists", "oil", "dax", "uschange", "elecdemand"]
for f in DATA_FILES:
    urllib.request.urlretrieve(f"{REPO}/data/{f}.csv", f"data/{f}.csv")

import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt
plt.rcParams.update({"figure.figsize": (9, 3.6), "axes.grid": True, "grid.alpha": 0.3, "axes.spines.top": False, "axes.spines.right": False})
import tsdata
print("Setup complete.")

### 11.1 Prices and returns

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy import stats
import tsdata
dax = tsdata.dax()                         # daily closing level of the German DAX index, 1991-1998 (1,860 trading days; base R EuStockMarkets)
r = (100 * np.log(dax).diff()).dropna(); r.name = 'log return (%)'
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
dax.plot(ax=axes[0], lw=0.9, title='DAX index level: a random walk with drift'); r.plot(ax=axes[1], lw=0.7, title='Daily log return (%): stationary, with bursts of volatility')
for ax in axes: ax.set_xlabel('trading day')
print(r.describe().round(3).to_string())
_caption = 'The level wanders; the returns hover around zero with clusters of large moves.'

### 11.2 Stylised facts of financial returns

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3))
plot_acf(r, lags=30, ax=axes[0], title='ACF of returns: no linear predictability'); plot_acf(r**2, lags=30, ax=axes[1], title='ACF of squared returns: volatility clustering')
axes[2].hist(r, bins=60, density=True, alpha=0.7, label='returns'); xs = np.linspace(r.min(), r.max(), 300); axes[2].plot(xs, stats.norm.pdf(xs, r.mean(), r.std()), color='#B8860B', lw=1.5, label='normal with same mean, s.d.')
axes[2].set_title('Fat tails'); axes[2].legend(fontsize=8); axes[2].set_yscale('log'); axes[2].set_ylim(1e-4, 1)
for ax in axes[:2]: ax.set_ylim(-0.3, 1)
lb_r, lb_r2 = acorr_ljungbox(r, lags=[10], return_df=True), acorr_ljungbox(r**2, lags=[10], return_df=True)
print(f'Ljung-Box(10): returns p = {lb_r.lb_pvalue.iloc[0]:.3f}   squared returns p = {lb_r2.lb_pvalue.iloc[0]:.4f}')
print(f'Excess kurtosis = {stats.kurtosis(r):.2f} (0 for normal); Jarque-Bera p = {stats.jarque_bera(r).pvalue:.2e}')
print(f'Largest |return| = {r.abs().max():.1f}%, i.e. {r.abs().max()/r.std():.1f} standard deviations; a normal would give a move this size about once in {1/(2*stats.norm.sf(r.abs().max()/r.std())):,.0f} days')
_caption = 'Three stylised facts on one index: uncorrelated returns, strongly autocorrelated squared returns, and tails far heavier than Gaussian (note the log scale).'

### 11.3 ARCH and GARCH

In [ ]:
from arch import arch_model
am = arch_model(r, mean='Constant', vol='GARCH', p=1, q=1, dist='t')
res = am.fit(disp='off')
print(res.summary().tables[1]); print(res.summary().tables[2]); print(res.summary().tables[3])
omega, alpha, beta = res.params['omega'], res.params['alpha[1]'], res.params['beta[1]']
print(f'\nalpha + beta = {alpha + beta:.3f} (persistence);  half-life of a variance shock = {np.log(0.5)/np.log(alpha + beta):.1f} days')
print(f'long-run daily variance = {omega/(1-alpha-beta):.3f}  ->  long-run daily volatility = {np.sqrt(omega/(1-alpha-beta)):.2f}%  ->  annualised ≈ {np.sqrt(252*omega/(1-alpha-beta)):.1f}%')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
r.plot(ax=axes[0], lw=0.6, title='Daily log return (%)'); axes[0].set_xlabel('')
res.conditional_volatility.plot(ax=axes[1], lw=1.2, label='GARCH(1,1)-t conditional volatility'); r.rolling(21).std().plot(ax=axes[1], lw=1, color='#B8860B', label='21-day rolling s.d.')
axes[1].axhline(np.sqrt(omega/(1-alpha-beta)), color='#555555', lw=0.8, ls='--', label='long-run level'); axes[1].legend(fontsize=8); axes[1].set_title('Volatility (%, daily)'); axes[1].set_xlabel('trading day')
_caption = 'The conditional volatility rises immediately after large returns and decays geometrically toward the long-run level; the rolling standard deviation lags and smooths.'

In [ ]:
z = res.std_resid.dropna()
lb1, lb2 = acorr_ljungbox(z, lags=[10], return_df=True), acorr_ljungbox(z**2, lags=[10], return_df=True)
print(f'Standardised residuals: Ljung-Box(10) p = {lb1.lb_pvalue.iloc[0]:.3f};  squared: p = {lb2.lb_pvalue.iloc[0]:.3f}  (both should be > 0.05)')
print(f'Excess kurtosis of standardised residuals = {stats.kurtosis(z):.2f}  (raw returns: {stats.kurtosis(r):.2f});  largest standardised residual = {z.abs().max():.1f}')

### 11.4 Forecasting volatility

In [ ]:
f = res.forecast(horizon=20, reindex=False)
vf = np.sqrt(f.variance.values.ravel())
print('Volatility forecast (%, daily) for the next 20 trading days:'); print(np.round(vf, 3))
print(f'converging toward the long-run level {np.sqrt(omega/(1-alpha-beta)):.2f}%; 10-day cumulative volatility = {np.sqrt(f.variance.values.ravel()[:10].sum()):.2f}%')

### 11.5 Worked example: GARCH on the simulated exchange-rate series

In [ ]:
fx = tsdata.nigeria_fx(); rfx = (100 * np.log(fx).diff()).dropna(); rfx.name = 'monthly log change (%)'
res_fx = arch_model(rfx, mean='Constant', vol='GARCH', p=1, q=1, dist='t').fit(disp='off')
print(res_fx.params.round(4).to_string())
print(f'\nalpha + beta = {res_fx.params["alpha[1]"] + res_fx.params["beta[1]"]:.3f}')

## Exercises

1. Show that the ten-day log return is the sum of ten daily log returns, and that the same is not true of simple returns. For daily returns of 1%, compute the difference after 250 days.
2. Derive the unconditional variance of GARCH(1,1) and the $h$-step variance forecast recursion.
3. Fit GARCH(1,1) with normal and Student-$t$ errors to the DAX returns; compare log-likelihoods and the standardised-residual kurtosis. Then fit GJR-GARCH and report whether the asymmetry parameter is significant.
4. Using the simulated `bonny_light()` monthly prices, compute log returns, test for ARCH effects, and fit GARCH(1,1). Is the oil price's volatility persistent? How does $\alpha + \beta$ compare with the equity case?

In [ ]:
# Your work here
